# 05 - SCD Type 2 Customer Dimension
Preserve historical customer attribute changes.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

base_path = '/mnt/retail'
source_path = f'{base_path}/input/customers.csv'
target_path = f'{base_path}/silver/dim_customer'
source = (spark.read.option('header', True).option('inferSchema', True).csv(source_path)
    .withColumn('effective_from', F.current_date())
    .withColumn('effective_to', F.lit(None).cast('date'))
    .withColumn('is_current', F.lit(True)))

In [ ]:
target = DeltaTable.forPath(spark, target_path)
(target.alias('t')
 .merge(source.alias('s'), 't.customer_code = s.customer_code AND t.is_current = true')
 .whenMatchedUpdate(
     condition='t.city <> s.city OR t.customer_name <> s.customer_name',
     set={'is_current': 'false', 'effective_to': 'current_date()'})
 .whenNotMatchedInsertAll()
 .execute())

### SCD Type 2 design
1. Detect changes. 2. Expire the old current row. 3. Insert the new version. 4. Preserve history.